In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from panel_methods.panel import Panel, SourceVortexPanelSystem
from panel_methods.elementary_flows import UniformFlow
from panel_methods.naca import naca4

In [ ]:
%matplotlib widget

# Variables
naca4_designation = '2412'
n_panels = 20 # Number of panels over top and bottom surface
blunt_te = False

# Generate panels
x_naca_upper, y_naca_upper, x_naca_lower, y_naca_lower = naca4(naca4_designation, n_panels + 1, blunt_te)
if blunt_te:
    x_naca = np.append(np.flip(x_naca_lower), np.append(x_naca_upper[1:], x_naca_lower[-1]))
    y_naca = np.append(np.flip(y_naca_lower), np.append(y_naca_upper[1:], y_naca_lower[-1]))
else:
    x_naca = np.append(np.flip(x_naca_lower), np.append(x_naca_upper[1:-1], x_naca_lower[-1]))
    y_naca = np.append(np.flip(y_naca_lower), np.append(y_naca_upper[1:-1], y_naca_lower[-1]))

panels = []
for i in range(len(x_naca) - 1):
    panel = Panel(x_naca[i], y_naca[i], x_naca[i + 1], y_naca[i + 1])
    panels.append(panel)

# Panel centres and normals
xc, yc = [panel.xc for panel in panels], [panel.yc for panel in panels]
xn, yn = [panel.xn for panel in panels], [panel.yn for panel in panels]

# Plot panels
padding = 0.25
plt.figure()
plt.plot(
    x_naca, y_naca,
    linewidth=2, linestyle='-', color='blue', label='Panels'
)
plt.plot(
    xc, yc,
    linestyle='none', color='blue',
    marker='o', markersize=5, markerfacecolor='black', markeredgecolor='black', label='Panel Centres'
)
plt.quiver(xc, yc, xn, yn, angles='xy', scale_units='xy', scale=10, label='Normals')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim([np.min(x_naca) - padding, np.max(x_naca) + padding])
plt.ylim([np.min(y_naca) - padding, np.max(y_naca) + padding])
plt.grid()
plt.legend()
plt.show()

In [ ]:
%matplotlib widget

# Variables
u_inf = 1.0
alpha = 2.0 * np.pi / 180.0

# Generate grid
N = 100
x_start, x_end = -1.0, 2.0
y_start, y_end = -0.5, 0.5
x_array = np.linspace(x_start, x_end, N)
y_array = np.linspace(y_start, y_end, N)
x_grid, y_grid = np.meshgrid(x_array, y_array)

# Create uniform flow
freestream = UniformFlow(u_inf, alpha)

# Create vortex panel system
source_vortex_panels = SourceVortexPanelSystem(panels, freestream)

# Solve
source_vortex_panels.solve()

# Velocity field
u_grid, v_grid = source_vortex_panels.velocity(x_grid, y_grid)

# Plot streamlines
plt.figure()
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.streamplot(x_grid, y_grid, u_grid, v_grid,
               density=2, linewidth=1, arrowsize=2, arrowstyle='->')
plt.plot(
    x_naca, y_naca,
    linewidth=2, linestyle='-', color='black', label='Panels'
)
plt.plot(
    xc, yc,
    linestyle='none', color='blue',
    marker='o', markersize=5, markerfacecolor='black', markeredgecolor='black', label='Panel Centres'
)
plt.grid()
plt.axis('scaled')
plt.legend()
plt.show()

# Compute pressure coefficient field
cp = 1.0 - (u_grid ** 2 + v_grid ** 2) / freestream.u_inf ** 2

# Plot pressure coefficient field
plt.figure()
plt.xlabel('x (m)')
plt.ylabel('y (m)')
contf = plt.contourf(x_grid, y_grid, cp,
                     levels=np.linspace(-2.0, 1.0, 100), extend='both')
cbar = plt.colorbar(contf)
cbar.set_label('$C_p$')
cbar.set_ticks([-2.0, -1.0, 0.0, 1.0])
plt.plot(
    x_naca, y_naca,
    linewidth=2, linestyle='-', color='black', label='Panels'
)
plt.plot(
    xc, yc,
    linestyle='none', color='blue',
    marker='o', markersize=5, markerfacecolor='black', markeredgecolor='black', label='Panel Centres'
)
plt.grid()
plt.axis('scaled')
plt.legend()
plt.show()

# Plot coefficient of pressure over cylinder surface
plt.figure()
plt.xlabel('x (m)')
plt.ylabel('$C_p$')
plt.plot(xc, source_vortex_panels.cp)
plt.grid()
plt.show()

# Calculate coefficient of lift and drag
cl = np.sum([-source_vortex_panels.cp[i] * panels[i].yn * panels[i].s for i in range(len(panels))])
cd = np.sum([-source_vortex_panels.cp[i] * panels[i].xn * panels[i].s for i in range(len(panels))])
print(f"Coefficient of lift: {cl}")
print(f"Coefficient of drag: {cd}")